# Decision Tree Regression (CART) — Google Colab

**Goal:** Predict a continuous target using **Decision Tree Regression (CART)** — captures **non-linear** patterns with easy-to-read rules.

| Example | Feature(s) (X) | Target (y) | Dataset |
|---------|----------------|------------|---------|
| **Example 1** | Level | Salary | `Datasets/Position_Salaries.csv` |
| **Example 2** | Area_sqft, Bedrooms, Age_years | Price | `Datasets/house_price.csv` |

| Phase | Topic | Cells |
|-------|-------|-------|
| Phase 0 | Setup | Install & Imports |
| — | Algorithm Guide | CART definitions, MSE, tree structure |
| Phase 1 | Data Pre-processing | Load → Cleaning → Encoding → Split |
| Phase 2 | Algorithm | Train → Predict → Visualize → Evaluate |

> **Run:** Runtime → Run all (or Ctrl+F9)

---
# Algorithm Guide — Decision Tree Regression (CART)

## What is CART?

**CART** = **Classification And Regression Trees**.  
For regression, the tree splits data to **minimize MSE** and predicts the **average** of y in each leaf.

## Tree Structure

| Part | Name | Role |
|------|------|------|
| **Root** | Root node | First split — top of the tree |
| **Internal node** | Branch | Decision: `feature ≤ threshold?` |
| **Leaf** | Terminal node | Final prediction = **mean(y)** of samples in that leaf |

## How Splitting Works (MSE Criterion)

At each node, CART tries every feature and threshold to find the split that **reduces MSE the most**:

**MSE (Mean Squared Error):**

`MSE = (1/n) · Σ(yᵢ − ȳ)²`

**Split score:** choose the split with the **largest MSE reduction** (greedy algorithm).

## Prediction Rule

1. Start at the **root** node.
2. Follow branches: `if X ≤ threshold` → left, else → right.
3. When you reach a **leaf**, predict `ŷ = mean(y)` of training samples in that leaf.

## Key Hyperparameters

| Parameter | Role | Effect |
|-----------|------|--------|
| `max_depth` | Max tree depth | Shallow = simpler, Deep = overfitting |
| `min_samples_split` | Min samples to split a node | Higher = simpler tree |
| `min_samples_leaf` | Min samples in a leaf | Higher = smoother predictions |
| `max_leaf_nodes` | Max number of leaves | Limits tree complexity |
| `random_state` | Random seed | Reproducible results |

## CART vs Other Models

| | Linear Regression | Decision Tree (CART) |
|---|-------------------|----------------------|
| Shape | Straight line / plane | **Step-like**, non-linear |
| Scaling | Sometimes needed | **Not required** |
| Interpretability | Coefficients | **Visual rules** (if tree is small) |
| Overfitting | Low (simple model) | **High** if tree is too deep |
| Multi-feature | Yes | Yes — splits on any feature |

## What the Student Must Remember

1. CART uses **MSE** to choose splits in regression.
2. Leaf prediction = **average** of y values in that region.
3. **No feature scaling** needed for Decision Trees.
4. Control **`max_depth`** to avoid overfitting.
5. Trees create **step functions** — predictions are constant within each region.

## Phase 0 — Cell 0: Install Libraries

Google Colab usually includes most libraries. This cell ensures required packages are available.

**What this cell does:** Installs scikit-learn, pandas, matplotlib, numpy, and seaborn quietly.

In [ ]:
# Install required libraries quietly (-q hides output)
!pip install -q scikit-learn pandas matplotlib numpy seaborn

## Phase 0 — Cell 1: Import Libraries

Import libraries for data handling, preprocessing, modeling, and evaluation.

**What this cell does:** Loads numpy, pandas, matplotlib, sklearn DecisionTreeRegressor, and metrics.

In [ ]:
# --- Import libraries ---
import numpy as np              # Numerical operations and arrays
import pandas as pd             # Load and manipulate tabular data
import matplotlib.pyplot as plt # Create charts and plots
import seaborn as sns           # Statistical visualizations (optional styling)

from sklearn.model_selection import train_test_split       # Split data into train/test
from sklearn.impute import SimpleImputer                   # Fill missing values
from sklearn.tree import DecisionTreeRegressor, plot_tree  # CART model and tree visualization
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Evaluation metrics

plt.rcParams['figure.figsize'] = (10, 6)  # Default plot size: width=10, height=6 inches
sns.set_theme(style='whitegrid')            # Clean white background with grid lines
np.random.seed(42)                          # Fix random seed for reproducible splits

print('Libraries ready')                      # Confirm all imports loaded successfully

---
# Example 1: Position Salaries — Decision Tree Regression

Predict **Salary** from job **Level**. The relationship is **non-linear** — CART creates step-like regions.

| Column | Role | Description |
|--------|------|-------------|
| `Level` | Feature (X) | Job level (1–10) |
| `Salary` | Target (y) | Annual salary in USD |

**File:** `Datasets/Position_Salaries.csv`

---
# Phase 1: Data Pre-processing

Prepare the data before training — same template is reused for other algorithms.

## Example 1 — Cell 1: Load and Explore Data

Load the CSV file and perform initial exploration (head, info, describe, shape).

**What this cell does:** Reads `Datasets/Position_Salaries.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('Datasets/Position_Salaries.csv')  # Read CSV into a DataFrame

FEATURE_COL = 'Level'   # Independent variable (X) — job level
TARGET_COL = 'Salary'   # Dependent variable (y) — salary to predict

print('First 5 rows:')          # Print a label for the table below
display(dataset.head())         # Show the first 5 rows to inspect the data

print('\nDataset info:')       # Print a label for column types and null counts
dataset.info()                  # Show column names, data types, and non-null counts

print('\nStatistical summary:')  # Print a label for numeric statistics
display(dataset.describe())       # Show count, mean, std, min, max, quartiles

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')  # Total rows and columns

## Example 1 — Cell 2: Data Cleaning (Handling Missing Values)

Check missing values, remove duplicates, and apply imputation if needed.

**What this cell does:** Cleans the dataset before modeling.

In [ ]:
# Step 2) Data cleaning

print('Missing values per column:')  # Print a label
print(dataset.isnull().sum())        # Count NaN values in each column

rows_before = len(dataset)                              # Store row count before cleaning
dataset = dataset.drop_duplicates().reset_index(drop=True)  # Remove duplicate rows
rows_after = len(dataset)                               # Store row count after deduplication
print(f'\nDuplicates removed: {rows_before - rows_after}')  # Show duplicate count

num_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()  # Numeric columns only
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')  # Fill NaN with column mean
if dataset.isnull().sum().sum() > 0:               # If missing values exist
    dataset[num_cols] = imputer.fit_transform(dataset[num_cols])  # Impute numeric columns
    print('Missing values imputed with mean')      # Confirm imputation
else:
    print('No missing values — imputer not applied')  # Skip when complete

print(f'\nRows after cleaning: {rows_after}')    # Final row count

## Example 1 — Cell 3: Categorical Data Encoding

We use `Level` (numeric). The `Position` column is text — skipped because Level already encodes rank.

**What this cell does:** Checks for categorical columns and encodes if needed.

In [ ]:
# Step 3) Categorical encoding

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()  # Find text columns
print(f'Categorical columns (not used as X): {cat_cols}')  # Position names — informational only
print(f'Feature used for CART: {FEATURE_COL}')              # Level is numeric — no encoding needed
print('No encoding required — X is numeric.')

## Example 1 — Cell 4: Splitting the Data

Define X (Level) and y (Salary), then split 80/20.

**What this cell does:** Creates feature/target arrays and applies train_test_split.

In [ ]:
# Step 4) Train-Test split

X = dataset[[FEATURE_COL]].values  # Feature matrix: Level (2D array for sklearn)
y = dataset[TARGET_COL].values     # Target vector: Salary values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,                  # Data to split
    test_size=0.2,         # 20% test, 80% train
    random_state=42        # Reproducible split
)

print(f'X_train shape: {X_train.shape}')  # Training features shape
print(f'X_test shape:  {X_test.shape}')   # Test features shape
print(f'y_train shape: {y_train.shape}')  # Training targets shape
print(f'y_test shape:  {y_test.shape}')   # Test targets shape

> **Note:** Decision Tree Regression does **not** require feature scaling. Trees split on thresholds — scale does not matter.

---
# Phase 2: Decision Tree Regression (CART)

Train a CART model to predict Salary from Level.

## Example 1 — Cell 5: Train the Model

Train `DecisionTreeRegressor` with `max_depth=4` to avoid overfitting on small data.

**What this cell does:** Fits the CART model and prints tree depth and leaf count.

In [ ]:
# Step 5) Train Decision Tree Regressor (CART)

regressor = DecisionTreeRegressor(
    max_depth=4,           # Limit depth to prevent overfitting on 10 samples
    min_samples_leaf=1,    # Minimum samples required in a leaf node
    random_state=42        # Reproducible tree structure
)

regressor.fit(X_train, y_train)  # Build tree: find best MSE splits on training data

print('Decision Tree (CART) trained successfully.')  # Confirm training complete
print(f'Tree depth: {regressor.get_depth()}')       # Actual depth of the built tree
print(f'Number of leaves: {regressor.get_n_leaves()}')  # Total leaf nodes (prediction regions)

## Example 1 — Cell 6: Predict

Predict Salary on the test set.

**What this cell does:** Generates predictions by routing each sample to a leaf.

In [ ]:
# Step 6) Predict

y_pred_train = regressor.predict(X_train)  # Predict Salary for training data
y_pred_test = regressor.predict(X_test)    # Predict Salary for test data

print('Sample predictions (Test set):')  # Print a label
for i in range(len(y_test)):             # Show all test predictions
    print(f'  Level={X_test[i][0]:.0f} -> Actual=${y_test[i]:,.0f}, Predicted=${y_pred_test[i]:,.0f}')

## Example 1 — Cell 7: Visualization

Plot the **step function** prediction curve and the **tree structure**.

**What this cell does:** Shows CART step predictions and a visual tree diagram.

In [ ]:
# Step 7) Visualization — step curve + tree diagram

fig, axes = plt.subplots(1, 2, figsize=(16, 6))  # Two subplots: curve and tree

# --- Left: step function curve ---
X_plot = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)  # 500 points for smooth steps
y_plot = regressor.predict(X_plot)                            # CART predictions (step function)

axes[0].scatter(X_train, y_train, color='blue', label='Training', s=80, zorder=3)  # Training points
axes[0].scatter(X_test, y_test, color='green', label='Test', s=80, zorder=3)       # Test points
axes[0].plot(X_plot, y_plot, color='red', linewidth=2, label='CART prediction')    # Step-like curve
axes[0].set_xlabel('Level')             # X-axis label
axes[0].set_ylabel('Salary (USD)')     # Y-axis label
axes[0].set_title('CART Step Function — Position Salaries')  # Subplot title
axes[0].legend()                        # Show legend

# --- Right: tree diagram ---
plot_tree(
    regressor,                          # Trained CART model
    feature_names=[FEATURE_COL],        # Name shown on split nodes
    filled=True,                        # Color nodes by value
    rounded=True,                       # Rounded node boxes
    fontsize=9,                         # Font size for readability
    ax=axes[1]                          # Draw on second subplot
)
axes[1].set_title('CART Tree Structure')  # Subplot title

plt.tight_layout()  # Adjust spacing
plt.show()          # Display both plots

## Example 1 — Cell 8: Evaluation

Evaluate CART with MAE, RMSE, and R² on the test set.

**What this cell does:** Computes and displays evaluation metrics.

In [ ]:
# Step 8) Evaluation
mae = mean_absolute_error(y_test, y_pred_test)              # Mean absolute error in USD
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))     # Root mean squared error
r2 = r2_score(y_test, y_pred_test)                          # Variance explained

results = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Value': [mae, rmse, r2],
    'Description': [
        'Mean Absolute Error (USD)',
        'Root Mean Squared Error (USD)',
        'Coefficient of Determination (1 = perfect)'
    ]
})

display(results.round(4))  # Show metrics table
print(f'\nExample 1 Test R² = {r2:.4f}')  # Print R² summary

## Why does CART work for Position Salaries?

| # | Reason | Explanation |
|---|--------|-------------|
| 1 | **Non-linear salary jumps** | Salary increases in steps at higher levels — tree splits match this |
| 2 | **No scaling needed** | Trees compare `Level ≤ threshold` — raw values work fine |
| 3 | **Step function output** | Each leaf predicts a constant salary — fits discrete jumps |
| 4 | **Visual rules** | `plot_tree` shows exact if/else rules students can read |
| 5 | **Watch overfitting** | With only 10 rows, use `max_depth` to limit tree size |

> **Summary:** CART creates **regions** of Level with constant salary predictions — ideal for step-like patterns.

---
# Example 2: House Price — Decision Tree Regression

Predict **Price** from three property features using CART with **multiple features**.

| Column | Role | Description |
|--------|------|-------------|
| `Area_sqft` | Feature (X₁) | Living area in square feet |
| `Bedrooms` | Feature (X₂) | Number of bedrooms |
| `Age_years` | Feature (X₃) | Age of the house in years |
| `Price` | Target (y) | Sale price in USD |

**File:** `Datasets/house_price.csv`

## Example 2 — Cell 1: Load and Explore Data

Load the house price CSV and inspect the data.

**What this cell does:** Reads `Datasets/house_price.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('Datasets/house_price.csv')  # Read CSV into a DataFrame

FEATURE_COLS = ['Area_sqft', 'Bedrooms', 'Age_years']  # Three input features
TARGET_COL = 'Price'                                    # Target variable

print('First 5 rows:')
display(dataset.head())

print('\nDataset info:')
dataset.info()

print('\nStatistical summary:')
display(dataset.describe())

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')

## Example 2 — Cell 2: Data Cleaning (Handling Missing Values)

This dataset includes missing values to demonstrate `SimpleImputer`.

**What this cell does:** Checks for nulls, removes duplicates, and imputes missing values.

In [ ]:
# Step 2) Data cleaning

print('Missing values per column:')
print(dataset.isnull().sum())

rows_before = len(dataset)
dataset = dataset.drop_duplicates().reset_index(drop=True)
rows_after = len(dataset)
print(f'\nDuplicates removed: {rows_before - rows_after}')

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
if dataset.isnull().sum().sum() > 0:
    dataset[FEATURE_COLS + [TARGET_COL]] = imputer.fit_transform(dataset[FEATURE_COLS + [TARGET_COL]])
    print('Missing values imputed with mean')
else:
    print('No missing values — imputer not applied')

print(f'\nRows after cleaning: {rows_after}')

## Example 2 — Cell 3: Categorical Data Encoding

All columns are numeric — encoding is skipped.

**What this cell does:** Confirms no categorical encoding is needed.

In [ ]:
# Step 3) Categorical encoding

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    print(f'Categorical columns found: {cat_cols}')
else:
    print('No categorical columns — encoding skipped.')
    print(f'Feature columns: {FEATURE_COLS}')

## Example 2 — Cell 4: Splitting the Data

Define X (3 features) and y (Price), then split 80/20.

**What this cell does:** Creates feature/target arrays and applies train_test_split.

In [ ]:
# Step 4) Train-Test split

X = dataset[FEATURE_COLS].values  # Feature matrix: 3 columns
y = dataset[TARGET_COL].values    # Target vector: Price values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train shape: {X_train.shape}')  # (n_train, 3)
print(f'X_test shape:  {X_test.shape}')   # (n_test, 3)
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')

## Example 2 — Cell 5: Train the Model

Train CART with multiple features — tree splits on Area, Bedrooms, or Age at each node.

**What this cell does:** Fits the model and reports tree structure.

In [ ]:
# Step 5) Train Decision Tree Regressor (CART)

regressor = DecisionTreeRegressor(
    max_depth=5,           # Limit depth for generalization
    min_samples_split=4,   # Need at least 4 samples to split a node
    min_samples_leaf=2,    # Each leaf must have at least 2 samples
    random_state=42
)

regressor.fit(X_train, y_train)  # Build tree using MSE criterion on all 3 features

print('Decision Tree (CART) trained successfully.')
print(f'Tree depth: {regressor.get_depth()}')
print(f'Number of leaves: {regressor.get_n_leaves()}')

print('\nFeature importances:')  # Which features were used most for splitting
for name, imp in zip(FEATURE_COLS, regressor.feature_importances_):
    print(f'  {name:15s} -> {imp:.4f}')  # Higher = more important for splits

## Example 2 — Cell 6: Predict

Predict house prices on the test set.

**What this cell does:** Generates predictions by routing samples through the tree.

In [ ]:
# Step 6) Predict

y_pred_train = regressor.predict(X_train)
y_pred_test = regressor.predict(X_test)

print('Sample predictions (Test set):')
for i in range(min(5, len(y_test))):
    print(f'  Actual=${y_test[i]:,.0f}, Predicted=${y_pred_test[i]:,.0f}')

## Example 2 — Cell 7: Visualization

Plot **Actual vs Predicted** and **feature importances**.

**What this cell does:** Visualizes model performance and which features matter most.

In [ ]:
# Step 7) Visualization

fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # Two subplots

axes[0].scatter(y_test, y_pred_test, color='green', alpha=0.7)  # Actual vs predicted
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price (USD)')
axes[0].set_ylabel('Predicted Price (USD)')
axes[0].set_title('Actual vs Predicted — House Price (CART)')
axes[0].legend()

axes[1].barh(FEATURE_COLS, regressor.feature_importances_, color='teal')  # Feature importance chart
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importances (CART)')

plt.tight_layout()
plt.show()

## Example 2 — Cell 8: Evaluation

Evaluate CART performance with MAE, RMSE, and R².

**What this cell does:** Computes and displays evaluation metrics.

In [ ]:
# Step 8) Evaluation
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

results = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Value': [mae, rmse, r2],
    'Description': [
        'Mean Absolute Error (USD)',
        'Root Mean Squared Error (USD)',
        'Coefficient of Determination (1 = perfect)'
    ]
})

display(results.round(4))
print(f'\nExample 2 Test R² = {r2:.4f}')

## Why does CART work well for House Price?

| # | Reason | Explanation |
|---|--------|-------------|
| 1 | **Multiple features** | CART splits on Area, Bedrooms, and Age automatically |
| 2 | **Non-linear interactions** | e.g. large Area + many Bedrooms → higher Price region |
| 3 | **Feature importances** | Shows which feature contributes most to splits |
| 4 | **No scaling needed** | Raw sqft, bedroom count, and age work directly |
| 5 | **High R² possible** | With tuned `max_depth`, CART fits complex price patterns |

> **Compare:** CART vs Linear Regression — trees capture **non-linear** and **interaction** effects without manual feature engineering.